In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
import io
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
import ipywidgets as widgets
from IPython.display import display, clear_output
import json
import torch.nn.functional as F
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from pathlib import Path
from IPython.display import Image as IPyImage, display


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
BASE_DIR = Path.cwd()
MODEL_DIR = BASE_DIR / "saved_models"
DATASET_DIR = BASE_DIR / "Dataset" / "LumbarSpinalStenosis" / "LumbarSpinalStenosis"

In [4]:
with open("label_map.json") as f:
    label_map = json.load(f)

idx_to_label = {v: k for k, v in label_map.items()}
class_names = list(label_map.keys())

label_explanations = {
    "Herniated Disc": "Disc material is displaced, possibly compressing nerves.",
    "No Stenosis": "Normal spinal canal with no narrowing detected.",
    "Thecal Sac": "Possible compression or narrowing around the thecal sac."
}

In [5]:
with open("test_metrics_summary.json") as f:
    test_metrics_summary = json.load(f)

# Convenience: pull out numbers for each model
cnn_test_acc  = test_metrics_summary["CNN"]["accuracy"]
cnn_test_bal  = test_metrics_summary["CNN"]["balanced_accuracy"]
res_test_acc  = test_metrics_summary["ResNet18"]["accuracy"]
res_test_bal  = test_metrics_summary["ResNet18"]["balanced_accuracy"]
eff_test_acc  = test_metrics_summary["EfficientNet-B0"]["accuracy"]
eff_test_bal  = test_metrics_summary["EfficientNet-B0"]["balanced_accuracy"]

In [6]:
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

In [7]:
class MRIDataset(Dataset):
    def __init__(self, df, transform=None, root_dir=DATASET_DIR):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.root_dir = Path(root_dir)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = self.root_dir / row["image_path"]
        image = Image.open(img_path).convert("L")
        label = label_map[row["label"]]

        if self.transform:
            image = self.transform(image)

        return image, label

In [8]:
val_df = pd.read_csv("val_split.csv")

val_dataset = MRIDataset(val_df, transform=transform)
val_loader = DataLoader(val_dataset, batch_size=32)

img, label = val_dataset[0]


In [9]:
def build_efficientnet():
    model = models.efficientnet_b0(weights=None)
    model.features[0][0] = nn.Conv2d(1, 32, 3, stride=2, padding=1, bias=False)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 3)
    return model

In [10]:
from pathlib import Path

BASE_DIR = Path.cwd()
MODEL_DIR = BASE_DIR / "saved_models"

effnet_path = MODEL_DIR / "efficientnet.pth"


In [11]:
# Load saved weights
effnet_loaded = build_efficientnet().to(device)
effnet_loaded.load_state_dict(torch.load("saved_models/efficientnet.pth", map_location=device))
effnet_loaded.eval()

loaded_models = {
    "EfficientNet-B0": effnet_loaded
}

In [12]:
#  Theme 
BG = "#F5F7FB"
CARD = "#FFFFFF"
CARD_SOFT = "#FAFBFF"
BORDER = "#E6EAF2"
TEXT = "#162033"
MUTED = "#667085"
ACCENT = "#7C6CF2"
ACCENT_2 = "#46B7A8"
ACCENT_3 = "#F28B82"
SHADOW = "0 10px 30px rgba(15, 23, 42, 0.06)"


In [13]:
# Helpers
def get_metrics(model, loader):
    model.eval()
    all_labels, all_preds = [], []

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            _, pred = torch.max(out, 1)

            all_labels.extend(y.cpu().numpy())
            all_preds.extend(pred.cpu().numpy())

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)

    acc = (all_labels == all_preds).mean() * 100
    bal_acc = balanced_accuracy_score(all_labels, all_preds) * 100
    rep = classification_report(all_labels, all_preds, target_names=class_names, digits=3)

    return acc, bal_acc, rep, all_labels, all_preds

In [14]:
def predict_uploaded_image(pil_img):
    model = effnet_loaded
    image = pil_img.convert("L")
    x = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(x)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()[0]

    pred_idx = int(np.argmax(probs))
    pred_label = idx_to_label[pred_idx]
    confidence = float(probs[pred_idx]) * 100
    explanation = label_explanations[pred_label]

    prob_dict = {
        idx_to_label[i]: float(probs[i])
        for i in range(len(probs))
    }

    return pred_label, confidence, prob_dict, explanation, image

In [15]:
def make_probability_figure(prob_dict):
    labels = list(prob_dict.keys())
    values = [prob_dict[k] * 100 for k in labels]

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=labels,
        y=values,
        text=[f"{v:.1f}%" for v in values],
        textposition="outside",
        marker=dict(color=[ACCENT_3, ACCENT_2, ACCENT]),
        hovertemplate="<b>%{x}</b><br>Confidence: %{y:.2f}%<extra></extra>"
    ))
    

    fig.update_layout(
        template="plotly_white",
        height=350,
        margin=dict(l=20, r=20, t=55, b=30),
        title=dict(text="Prediction Confidence", x=0.03, font=dict(size=18, color=TEXT)),
        paper_bgcolor=CARD,
        plot_bgcolor=CARD,
        font=dict(family="Arial", size=13, color=TEXT),
        yaxis=dict(
            title="Probability (%)",
            range=[0, 100],
            gridcolor="rgba(124,108,242,0.10)",
            zeroline=False
        ),
        xaxis=dict(title=None),
        showlegend=False
    )
    return fig


In [16]:
def make_comparison_figure():
    cnn_acc, _, _, _, _ = get_metrics(loaded_models["CNN"], val_loader)
    res_acc, _, _, _, _ = get_metrics(loaded_models["ResNet18"], val_loader)
    eff_acc, _, _, _, _ = get_metrics(loaded_models["EfficientNet-B0"], val_loader)

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=["CNN", "ResNet18", "EfficientNet-B0"],
        y=[cnn_acc, res_acc, eff_acc],
        text=[f"{cnn_acc:.2f}%", f"{res_acc:.2f}%", f"{eff_acc:.2f}%"],
        textposition="outside",
        marker=dict(color=[ACCENT_3, ACCENT_2, ACCENT]),
        hovertemplate="<b>%{x}</b><br>Accuracy: %{y:.2f}%<extra></extra>"
    ))

    fig.update_layout(
        template="plotly_white",
        height=420,
        width=920,
        margin=dict(l=20, r=20, t=55, b=30),
        title=dict(text="Validation Accuracy Comparison", x=0.03, font=dict(size=18, color=TEXT)),
        paper_bgcolor=CARD,
        plot_bgcolor=CARD,
        font=dict(family="Arial", size=13, color=TEXT),
        yaxis=dict(
            title="Accuracy (%)",
            gridcolor="rgba(124,108,242,0.10)",
            zeroline=False
        ),
        xaxis=dict(title=None),
        showlegend=False
    )
    return fig

In [17]:
def plot_confusion_matrix(labels, preds, title, cmap="Blues"):
    cm = confusion_matrix(labels, preds, normalize="true")

    plt.figure(figsize=(4.8, 4.2))
    sns.heatmap(
        cm,
        annot=True,
        fmt=".2f",
        cmap=cmap,
        xticklabels=class_names,
        yticklabels=class_names
    )
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

In [18]:
# cards 
dashboard_shell = widgets.HTML(
    value=f"""
    <div style="
        background:{BG};
        padding:26px;
        border-radius:28px;
        border:1px solid {BORDER};
        box-shadow:{SHADOW};
        margin-bottom:18px;
    ">
        <div style="
            background:linear-gradient(135deg, #F2F0FF 0%, #F8FAFF 45%, #EDFCFA 100%);
            border:1px solid {BORDER};
            border-radius:24px;
            padding:28px 30px;
        ">
            <div style="font-size:12px; font-weight:700; letter-spacing:1.4px; color:{ACCENT}; text-transform:uppercase;">
                Clinical Machine Learning Model
            </div>
            <h1 style="margin:10px 0 12px 0; color:{TEXT}; font-size:32px; font-weight:700;">
                Lumbar Spinal Stenosis MRI Dashboard
            </h1>
            <p style="margin:0; color:{MUTED}; font-size:15px; line-height:1.65; max-width:780px;">
                A refined interactive dashboard for spinal MRI classification using EfficientNet-B0.
            </p>
        </div>
    </div>
    """
)

predict_header = widgets.HTML(
    value=f"""
    <div style="background:{CARD}; border:1px solid {BORDER}; border-radius:22px; padding:18px 20px; margin-bottom:12px; box-shadow:{SHADOW};">
        <h3 style="margin:0; color:{TEXT}; font-size:20px;">Upload & Predict</h3>
        <p style="margin:6px 0 0 0; color:{MUTED}; font-size:13px;">
            Upload an MRI scan and generate a prediction with model confidence and clinical-style label explanation.
        </p>
    </div>
    """
)

comparison_header = widgets.HTML(
    value=f"""
    <div style="background:{CARD}; border:1px solid {BORDER}; border-radius:22px; padding:18px 20px; margin-bottom:12px; box-shadow:{SHADOW};">
        <h3 style="margin:0; color:{TEXT}; font-size:20px;">Model Insights</h3>
        <p style="margin:6px 0 0 0; color:{MUTED}; font-size:13px;">
            Review comparative validation performance, class-level behavior, and overall model reliability.
        </p>
    </div>
    """
)

upload_disclaimer = widgets.HTML(
    value=f"""
    <div style="
        background:linear-gradient(135deg,#FFF8E8,#FFFDF4);
        color:#9A6700;
        border:1px solid #F6D28D;
        border-radius:18px;
        padding:14px 16px;
        margin:8px 0 16px 0;
        font-size:13px;
        line-height:1.6;
    ">
        <b>Disclaimer:</b> EfficientNet-B0 is used for uploaded MRI prediction.
        This dashboard is intended for educational demonstration only.
    </div>
    """
)

In [19]:
prediction_html = widgets.HTML()
kpi_cards = widgets.HTML()

def update_kpi_cards():
    kpi_cards.value = f"""
    <div style='display:flex; gap:16px; flex-wrap:wrap; margin:8px 0 16px 0;'>
        <div style='flex:1; min-width:220px; background:{CARD}; border:1px solid {BORDER}; border-radius:22px; padding:20px; box-shadow:{SHADOW};'>
            <div style='font-size:12px; color:{MUTED}; text-transform:uppercase; letter-spacing:1px;'>CNN (Test)</div>
            <div style='font-size:30px; font-weight:700; color:{TEXT}; margin-top:8px;'>{cnn_test_acc:.2f}%</div>
            <div style='margin-top:8px; color:{MUTED}; font-size:13px;'>Balanced Accuracy: <b>{cnn_test_bal:.2f}%</b></div>
        </div>

        <div style='flex:1; min-width:220px; background:{CARD}; border:1px solid {BORDER}; border-radius:22px; padding:20px; box-shadow:{SHADOW};'>
            <div style='font-size:12px; color:{MUTED}; text-transform:uppercase; letter-spacing:1px;'>ResNet18 (Test)</div>
            <div style='font-size:30px; font-weight:700; color:{TEXT}; margin-top:8px;'>{res_test_acc:.2f}%</div>
            <div style='margin-top:8px; color:{MUTED}; font-size:13px;'>Balanced Accuracy: <b>{res_test_bal:.2f}%</b></div>
        </div>

        <div style='flex:1; min-width:220px; background:{CARD}; border:1px solid {BORDER}; border-radius:22px; padding:20px; box-shadow:{SHADOW};'>
            <div style='font-size:12px; color:{MUTED}; text-transform:uppercase; letter-spacing:1px;'>EfficientNet-B0 (Test)</div>
            <div style='font-size:30px; font-weight:700; color:{TEXT}; margin-top:8px;'>{eff_test_acc:.2f}%</div>
            <div style='margin-top:8px; color:{MUTED}; font-size:13px;'>Balanced Accuracy: <b>{eff_test_bal:.2f}%</b></div>
        </div>
    </div>
    """

In [20]:
# Widgets
upload_widget = widgets.FileUpload(
    accept="image/*",
    multiple=False,
    description="Upload MRI",
    layout=widgets.Layout(width="220px")
)

predict_button = widgets.Button(
    description="Run Prediction",
    button_style="success",
    layout=widgets.Layout(width="180px", height="42px")
)
predict_button.style.font_weight = "600"

upload_image_output = widgets.Output(
    layout=widgets.Layout(border=f'1px solid {BORDER}', padding='16px', width='48%')
)
prob_output = widgets.Output(
    layout=widgets.Layout(border=f'1px solid {BORDER}', padding='16px', width='48%')
)
comparison_output = widgets.Output(
    layout=widgets.Layout(border=f'1px solid {BORDER}', padding='16px')
)
metrics_output = widgets.Output(
    layout=widgets.Layout(border=f'1px solid {BORDER}', padding='16px')
)


In [21]:
def run_uploaded_prediction(button=None):
    with upload_image_output:
        clear_output(wait=True)

    with prob_output:
        clear_output(wait=True)

    prediction_html.value = ""

    if not upload_widget.value:
        prediction_html.value = f"""
        <div style='background:#FEF2F2; color:#B91C1C; border:1px solid #FECACA;
                    border-radius:18px; padding:14px; margin-top:8px; font-weight:600;'>
            Please upload an MRI image first.
        </div>
        """
        return

    try:
        uploaded_file = upload_widget.value[0]
        content = uploaded_file["content"]

        if hasattr(content, "tobytes"):
            content = content.tobytes()

        pil_img = Image.open(io.BytesIO(content)).convert("RGB")

        pred_label, confidence, prob_dict, explanation, gray_img = predict_uploaded_image(
            pil_img
        )

        with upload_image_output:
            plt.figure(figsize=(4, 4))
            plt.imshow(gray_img, cmap="gray")
            plt.axis("off")
            plt.title("Uploaded MRI", color=TEXT)
            plt.show()

        prediction_html.value = f"""
        <div style='margin-top:14px; background:{CARD}; border:1px solid {BORDER};
                    border-radius:22px; padding:20px; box-shadow:{SHADOW};'>
            <div style='font-size:12px; color:{ACCENT}; text-transform:uppercase; letter-spacing:1px;'>Prediction Result</div>
            <h3 style='margin:10px 0 12px 0; color:{TEXT}; font-size:26px;'>{pred_label}</h3>
            <p style='margin:0 0 8px 0; color:{TEXT}; font-size:14px;'><b>Confidence Score:</b> {confidence:.2f}%</p>
            <p style='margin:0 0 8px 0; color:{MUTED}; font-size:14px; line-height:1.6;'><b>Explanation:</b> {explanation}</p>
        </div>
        """

        with prob_output:
            fig = make_probability_figure(prob_dict)
            fig.show()

    except Exception as e:
        prediction_html.value = f"""
        <div style='background:#FEF2F2; color:#B91C1C; border:1px solid #FECACA;
                    border-radius:18px; padding:14px; margin-top:8px; font-weight:600;'>
            Error while running prediction: {str(e)}
        </div>
        """

In [22]:
def make_comparison_figure():
    models_list = ["CNN", "ResNet18", "EfficientNet-B0"]
    accs = [cnn_test_acc, res_test_acc, eff_test_acc]

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=models_list,
        y=accs,
        text=[f"{a:.2f}%" for a in accs],
        textposition="outside",
        marker=dict(color=[ACCENT_3, ACCENT_2, ACCENT]),
        hovertemplate="<b>%{x}</b><br>Test Accuracy: %{y:.2f}%<extra></extra>"
    ))

    fig.update_layout(
        template="plotly_white",
        height=420,
        width=920,
        margin=dict(l=20, r=20, t=55, b=30),
        title=dict(text="Test Accuracy Comparison", x=0.03, font=dict(size=18, color=TEXT)),
        paper_bgcolor=CARD,
        plot_bgcolor=CARD,
        font=dict(family="Arial", size=13, color=TEXT),
        yaxis=dict(
            title="Accuracy (%)",
            gridcolor="rgba(124,108,242,0.10)",
            zeroline=False
        ),
        xaxis=dict(title=None),
        showlegend=False
    )
    return fig

In [23]:
predict_button.on_click(run_uploaded_prediction)

In [24]:
# Comparison tab
def show_comparison():
    with comparison_output:
        clear_output(wait=True)
        fig = make_comparison_figure()
        fig.show()

In [25]:
def show_metrics():
    with metrics_output:
        clear_output(wait=True)

        # Load metrics JSON
        reports_path = BASE_DIR / "test_metrics_summary.json"
        with open(reports_path, "r") as f:
            reports_dict = json.load(f)

        cm_files = [
            ("CNN", "cnn_test_confusion.png"),
            ("ResNet18", "resnet18_test_confusion.png"),
            ("EfficientNet-B0", "efficientnet_test_confusion.png")
        ]

        for model_name, file_name in cm_files:
            # Header card
            display(widgets.HTML(
                value=f"""
                <div style='margin:14px 0 10px 0; background:{CARD_SOFT}; border:1px solid {BORDER};
                            border-radius:20px; padding:18px; box-shadow:{SHADOW};'>
                    <div style='font-size:12px; color:{ACCENT}; text-transform:uppercase; letter-spacing:1px;'>
                        {model_name}
                    </div>
                    <h4 style='margin:8px 0 4px 0; color:{TEXT}; font-size:20px;'>
                        Test Confusion Matrix & Report
                    </h4>
                </div>
                """
            ))

            # Confusion matrix image
            file_path = BASE_DIR / file_name
            if file_path.exists():
                display(IPyImage(filename=str(file_path)))
            else:
                display(widgets.HTML(
                    value=f"""
                    <div style='background:#FEF2F2; color:#B91C1C; border:1px solid #FECACA;
                                border-radius:16px; padding:10px; margin:8px 0; font-size:13px;'>
                        Confusion matrix image not found: {file_name}
                    </div>
                    """
                ))

            # Metrics from JSON
            model_metrics = reports_dict.get(model_name)

            if model_metrics:
                acc = model_metrics.get("accuracy", 0)
                bal_acc = model_metrics.get("balanced_accuracy", 0)
                cr = model_metrics.get("classification_report", {})

                rows = []
                class_order = ["Herniated Disc", "No Stenosis", "Thecal Sac"]

                for cls in class_order:
                    if cls in cr:
                        s = cr[cls]
                        rows.append(f"""
                        <tr>
                            <td style="padding:8px; border-bottom:1px solid {BORDER};">{cls}</td>
                            <td style="padding:8px; border-bottom:1px solid {BORDER}; text-align:right;">{s['precision']:.3f}</td>
                            <td style="padding:8px; border-bottom:1px solid {BORDER}; text-align:right;">{s['recall']:.3f}</td>
                            <td style="padding:8px; border-bottom:1px solid {BORDER}; text-align:right;">{s['f1-score']:.3f}</td>
                            <td style="padding:8px; border-bottom:1px solid {BORDER}; text-align:right;">{int(s['support'])}</td>
                        </tr>
                        """)

                for avg_name in ["macro avg", "weighted avg"]:
                    if avg_name in cr:
                        s = cr[avg_name]
                        rows.append(f"""
                        <tr style="background:{CARD_SOFT}; font-weight:600;">
                            <td style="padding:8px; border-bottom:1px solid {BORDER};">{avg_name}</td>
                            <td style="padding:8px; border-bottom:1px solid {BORDER}; text-align:right;">{s['precision']:.3f}</td>
                            <td style="padding:8px; border-bottom:1px solid {BORDER}; text-align:right;">{s['recall']:.3f}</td>
                            <td style="padding:8px; border-bottom:1px solid {BORDER}; text-align:right;">{s['f1-score']:.3f}</td>
                            <td style="padding:8px; border-bottom:1px solid {BORDER}; text-align:right;">{int(s['support'])}</td>
                        </tr>
                        """)

                display(widgets.HTML(
                    value=f"""
                    <div style='background:{CARD}; border:1px solid {BORDER}; border-radius:16px;
                                padding:14px 16px; margin:8px 0 16px 0;'>
                        <p style='margin:0 0 10px 0; color:{TEXT}; font-size:14px;'>
                            <b>Accuracy:</b> {acc:.2f}% &nbsp;&nbsp; | &nbsp;&nbsp;
                            <b>Balanced Accuracy:</b> {bal_acc:.2f}%
                        </p>

                        <table style="width:100%; border-collapse:collapse; font-size:13px; color:{TEXT};">
                            <thead>
                                <tr style="background:{CARD_SOFT};">
                                    <th style="padding:8px; text-align:left; border-bottom:1px solid {BORDER};">Class</th>
                                    <th style="padding:8px; text-align:right; border-bottom:1px solid {BORDER};">Precision</th>
                                    <th style="padding:8px; text-align:right; border-bottom:1px solid {BORDER};">Recall</th>
                                    <th style="padding:8px; text-align:right; border-bottom:1px solid {BORDER};">F1-score</th>
                                    <th style="padding:8px; text-align:right; border-bottom:1px solid {BORDER};">Support</th>
                                </tr>
                            </thead>
                            <tbody>
                                {''.join(rows)}
                            </tbody>
                        </table>
                    </div>
                    """
                ))
            else:
                display(widgets.HTML(
                    value=f"""
                    <div style='background:#FFF4E5; color:#9A6700; border:1px solid #F6D28D;
                                border-radius:16px; padding:10px; margin:8px 0 16px 0; font-size:13px;'>
                        No classification report found in JSON for <b>{model_name}</b>.
                    </div>
                    """
                ))

In [26]:
# Layout 
control_row = widgets.HBox(
    [ upload_widget, predict_button],
    layout=widgets.Layout(gap="12px", align_items="center", margin="0 0 10px 0")
)

predict_visual_row = widgets.HBox(
    [upload_image_output, prob_output],
    layout=widgets.Layout(justify_content="space-between", gap="14px")
)

tab1 = widgets.VBox([
    dashboard_shell,
    predict_header,
    upload_disclaimer,
    control_row,
    prediction_html,
    predict_visual_row
])

tab2 = widgets.VBox([
    comparison_header,
    kpi_cards,
    comparison_output,
    metrics_output
])

tab = widgets.Tab(children=[tab1, tab2])
tab.set_title(0, "Predict")
tab.set_title(1, "Model Insights")

display(tab)

update_kpi_cards()
show_comparison()
show_metrics()